In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

train_df = pd.read_csv('train_preprocessed.csv')
test_df = pd.read_csv('test_preprocessed.csv')

X_train = train_df.drop('RainTomorrow', axis=1)
y_train = train_df['RainTomorrow']
X_test = test_df.drop('RainTomorrow', axis=1)
y_test = test_df['RainTomorrow']

X_tune = X_train.sample(frac=1, random_state=42)
y_tune = y_train.loc[X_tune.index]

knn_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('knn', KNeighborsClassifier(n_jobs=-1))
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 11, 15, 21],
    'knn__weights': ['uniform', 'distance']
}

search = GridSearchCV(
    knn_pipeline, 
    param_grid=param_grid, 
    cv=3, 
    scoring='f1',
    n_jobs=-1
)
search.fit(X_tune, y_tune)

results = pd.DataFrame(search.cv_results_)
cols = ['param_knn__n_neighbors', 'param_knn__weights', 'mean_test_score', 'rank_test_score']
report_df = results[cols].sort_values(by='mean_test_score', ascending=False)
report_df.columns = ['n_neighbors', 'weights', 'avg_F1-Score', 'rank']
print(report_df.to_string(index=False))
best_knn = search.best_estimator_

best_knn.fit(X_train, y_train)

y_pred = best_knn.predict(X_test)
y_pred_proba = best_knn.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No Rain (0)', 'Rain (1)']))

auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {auc_score:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))